# Jointure de tables

## Note préliminaire sur les données

Les données étant volumineuses (1,5 Go une fois chargées dans une feuille de calcul), nous allons définir une ou plusieurs fonctions par exercice pour créer nos diagrammes.

De cette manière, à la sortie de chacune des fonctions, les variables temporaires utilisées seront supprimées et la mémoire pourra être réutilisée.

## Imports

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import pandas
import seaborn as sns

## Chargement des données

Pour ces travaux pratiques, nous allons utiliser des données de transactions immobilières sur la France entière, entre 2014 et 2022.

Ces données proviennent du site d'[open data français](https://www.data.gouv.fr/fr/datasets/demandes-de-valeurs-foncieres-geolocalisees/). Nous utiliserons là une version retravaillée qui regroupe les transactions, qui provient du dépôt [NyxAether/DVF](https://github.com/NyxAether/DVF) sur GitHub.

In [ ]:
!git clone https://github.com/mlambda/dataset-dvf.git

In [ ]:
df = pandas.read_parquet("dataset-dvf/dvf-linearized-2014-2022.parquet")

In [ ]:
df["date"] = pandas.to_datetime(
    dict(year=df.annee_mutation, month=df.mois_mutation, day=df.jour_mutation)
)
df.drop(columns=["annee_mutation", "mois_mutation", "jour_mutation"], inplace=True)

In [ ]:
# Pour une raison de lisibilité, on garde le 0 devant les départements à un chiffre
# replacedict = {"code_departement":{x:"0"+x for x in df.code_departement.unique() if len(x) == 1}}
# df.replace(replacedict,inplace=True)

replacedict = {x: "0" + x for x in df.code_departement.unique() if len(x) == 1}
df["code_departement"] = df["code_departement"].cat.rename_categories(replacedict)

In [ ]:
df.info()

## Valeur foncière totale par région et par an

Nous allons étudier dans cette partie comment décliner une variable catégorielle en fonction d'une autre variable : nous allons produire un diagramme de la valeur foncière totale changée par région, et cette valeur sera elle-même déclinée par année.

Pour cela, commençons par récupérer un jeu de données qui nous permettra de regrouper les départements en régions :

In [ ]:
!wget https://www.data.gouv.fr/fr/datasets/r/987227fb-dcb2-429e-96af-8979f97c9c84 -O regions.csv

Utilisez ce nouveau jeu de donnée pour faire une jointure avec notre jeu original.
- *Chargez `regions.csv` dans un `DataFrame`*
- *Ajoutez une colonne qui contient le nom de région dans votre feuille de données de travail, grâce à la fonction [`pandas.DataFrame.join`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.join.html).*
- *Regroupez les données par région puis par année à l'aide de la fonction [`pandas.DataFrame.groupby`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html).*
- *Utilisez la fonction [`seaborn.barplot`](https://seaborn.pydata.org/generated/seaborn.barplot.html) pour afficher ces données regroupées. Vous pouvez vous référer à la partie du [tutoriel de seaborn](https://seaborn.pydata.org/tutorial/introduction.html#plots-for-categorical-data) qui traite des données catégorielles complexes.*

In [ ]:
def value_by_region_by_year() -> None:
    pass  # Votre code ici


value_by_region_by_year()

### Solution

In [ ]:
def get_regions_df() -> pandas.DataFrame:
    df = pandas.read_csv(
        "regions.csv",
        index_col="num_dep",
        usecols=["num_dep", "region_name"],
        dtype=dict(region_name="category"),
    )
    return df


get_regions_df()

In [ ]:
def value_by_region_by_year() -> None:
    # jointure des tables
    joined = df.join(get_regions_df(), on="code_departement")
    # groupby par région et par date, aggrégation par somme des valeur foncières
    data = (
        joined.groupby([joined.region_name, joined.date.dt.year])
        .valeur_fonciere.sum()
        .reset_index()
    )

    # On veut afficher les résultats par valeur fonciere totale de région décroissante
    sort_index = (
        joined.groupby(joined.region_name)
        .valeur_fonciere.sum()
        .sort_values(ascending=False)
        .index
    )
    # affichage
    fig, ax = plt.subplots(figsize=(20, 10))
    sns.barplot(
        data, x="region_name", y="valeur_fonciere", hue="date", order=sort_index, ax=ax
    )
    sns.despine(fig)
    ax.set_title("Montant total des transactions par département et par année")
    ax.set_xlabel("Région")
    plt.xticks(rotation=45)
    ax.set_ylabel("Montant total des transactions")
    ax.get_yaxis().set_major_formatter(
        matplotlib.ticker.EngFormatter(unit="€", places=1)
    )
    fig.show()


value_by_region_by_year()

## Ensoleillement

En utilisant les deux fichier de l'opendata français suivant [ensoleillement.csv](https://www.data.gouv.fr/fr/datasets/r/4cf82433-9053-4ea8-87d7-c8d409dd7bb7) et [departements.csv](https://www.data.gouv.fr/fr/datasets/r/70cef74f-70b1-495a-8500-c089229c0254)

Augmentez l'information de notre jeu de donnée original.


In [ ]:
! wget https://www.data.gouv.fr/fr/datasets/r/4cf82433-9053-4ea8-87d7-c8d409dd7bb7 -O ensoleillements.csv
! wget https://www.data.gouv.fr/fr/datasets/r/70cef74f-70b1-495a-8500-c089229c0254 -O departements.csv

In [ ]:
def ensoleillement() -> None:
    pass  # Votre code ici


ensoleillement()

### Solution

In [ ]:
def ensoleillement() -> None:
    sol = pandas.read_csv("ensoleillements.csv")
    dep = pandas.read_csv(
        "departements.csv", usecols=["code_departement", "nom_departement"]
    )
    sol = dep.join(sol)[["code_departement", "Temps d'enseillement (jours/an)"]]
    sol["code_departement"].astype("category")
    return df.merge(sol, on="code_departement", how="left")


ensoleillement()